# 📖 教案：App4 多智能体协作 — 让多个 AI “组队打怪”

---

## 课程概览

| 项目 | 内容 |
|:---|:---|
| **课程名称** | 多智能体协作系统：从消息协议到流水线+辩论模式 |
| **对应源码** | `Applications/App4_Multi_Agent.ipynb`（共 30 个 Cell） |
| **总时长** | 约 80 分钟（含 2 次休息） |
| **目标受众** | 有 Python 基础、了解 LLM 基本概念的学员 |
| **前置知识** | Ch12（Agent 原理）、App1（ReAct Agent / Function Calling） |
| **环境要求** | Ollama 本地后端（`qwen3:4b`）或 DashScope API Key |

---

## ⏱ 时间表

| 时间段 | 时长 | 内容 | 对应 Cell |
|:---|:---|:---|:---|
| 00:00–00:08 | 8 min | 开场：为什么需要多智能体？四大模式概览 | Cell 0–1 |
| 00:08–00:15 | 7 min | 环境配置与 Ollama 后端确认 | Cell 2–4 |
| 00:15–00:30 | 15 min | Part 1：消息协议 + BaseAgent 基类 | Cell 5–8 |
| 00:30–00:33 | 3 min | ☕ 第一次休息 | — |
| 00:33–00:50 | 17 min | Part 2：四大专业化 Agent（Planner/Coder/Reviewer/Tester） | Cell 9–13 |
| 00:50–01:00 | 10 min | Part 3：编排器 MultiAgentOrchestrator | Cell 14–15 |
| 01:00–01:15 | 15 min | Part 4：运行流水线 + 结果分析 | Cell 16–22 |
| 01:15–01:18 | 3 min | ☕ 第二次休息 | — |
| 01:18–01:30 | 12 min | Part 5：辩论模式（DebateAgent + Moderator） | Cell 23–26 |
| 01:30–01:38 | 8 min | Part 6：可视化 + 总结 | Cell 27–29 |

---

## ✅ 课前检查清单

- [ ] Ollama 服务已启动（`ollama serve`）或 DashScope API Key 已配置
- [ ] `qwen3:4b` 模型已拉取（`ollama pull qwen3:4b`）
- [ ] Python 环境中已安装 `matplotlib`、`numpy`
- [ ] 已完成 `PREPARE_OLLAMA.ipynb` 准备步骤
- [ ] 投影/屏幕共享已就绪，字体放大到可读
- [ ] 提前跑通一遍 Cell 17–21，确认 LLM 调用正常（每轮约 5–15 秒）

---

# 第一部分：开场与概念引入（00:00–00:08）

---

## 教学段 1：为什么需要多智能体？

📍 **Cell 范围**：Cell 0（`cell-0`）、Cell 1（`cell-1`）

⏱ **时间**：8 分钟

🎯 **目标**：让学员理解“单个 Agent 能力有限，多个专业化 Agent 协作才能解决复杂问题”，建立直觉

---

### 🗣 话术

> 大家好！今天我们来搞一个特别有意思的项目——让多个 AI 组队干活。
>
> 先问大家：**如果你是一个创业公司的 CTO，你会雇一个全栈工程师什么都干，还是组一个团队——一个做产品规划、一个写代码、一个做代码审查、一个写测试？**（等 2 秒）
>
> 对吧，复杂的项目一定是团队协作。道理很简单：**一个人再厉害，也不可能同时精通所有领域。**
>
> AI Agent 也是一样。在 App1 里，我们的 ReAct Agent 什么都自己干——推理、调工具、总结结果。对于简单任务没问题，但如果任务变成“写一个完整的函数，要有代码审查和单元测试”，一个 Agent 就容易顾此失彼。
>
> 看 Cell 0 里列的四大模式：
> - **监督者模式**：一个领导协调大家，像项目经理
> - **辩论模式**：多个 Agent 从不同角度争论，像圆桌讨论
> - **流水线模式**：一个做完传给下一个，像工厂流水线
> - **群体模式**：自由协作，像开源社区
>
> 今天我们重点实现两种：**流水线模式**（Planner → Coder → Reviewer → Tester）和**辩论模式**。
>
> 大家看 Cell 1 的学习目标，一共四个，50 分钟的代码量其实不多，关键是理解**设计思路**。

---

### 👀 输出要点

- 学员能说出多智能体协作的核心优势：分工专注、相互质检、可扩展
- 理解“软件团队”类比：Planner=PM、Coder=工程师、Reviewer=审查员、Tester=QA
- 明确本课两大实践目标：流水线模式 + 辩论模式

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 多智能体和多轮对话有什么区别？ | 多轮对话是一个 Agent 和用户来回聊；多智能体是多个 Agent 之间互相发消息协作。Agent 之间有明确的角色分工和通信协议。 |
| 为什么不让一个强大的模型干所有事？ | 第一，单模型容易“角色混乱”（又要写代码又要审查自己）；第二，分工后每个 Agent 的 Prompt 更聚焦、输出更可控；第三，可以独立升级或替换某个 Agent。 |
| 和 AutoGen / CrewAI 等框架的关系？ | 那些是生产级框架，核心原理和我们今天手写的一样——消息传递、角色 Prompt、编排器。理解原理后用框架才不是黑盒。 |

---

### ➡️ 转场

> 好，概念到位了。先把环境跑起来，确保 LLM 后端就绪。

---

## 教学段 2：环境配置（00:08–00:15）

📍 **Cell 范围**：Cell 2（`7770a35c`）、Cell 3（`cell-3`）、Cell 4（`cell-4`）

⏱ **时间**：7 分钟

🎯 **目标**：跑通环境导入，确认 LLM 后端可用

---

### 🗣 话术

> 来，先运行 Cell 4。这个 Cell 做三件事：
> 1. 导入所有依赖——`dataclass`、`Enum`、`ABC` 这些都是 Python 标准库，不需要额外安装
> 2. 导入我们自己封装的 `get_llm_backend`（和 App1 一样的后端工具）
> 3. 配置 matplotlib 中文字体
>
> **注意**：如果要用 DashScope API，在 `_dashscope_key` 那里填上你的 Key。留空就默认走 Ollama 本地。
>
> 运行后应该没有报错输出。如果报错 `ModuleNotFoundError`，检查你的 Python 环境是不是对的。

---

### 👀 输出要点

- Cell 4 运行无报错即可
- 如果有 `ImportError`，说明环境未正确配置，参考 `PREPARE_OLLAMA.ipynb`

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| `sys.path.insert` 是干什么的？ | 把项目根目录加入 Python 搜索路径，这样才能 `from utils.llm_backend import ...`。 |
| 为什么导入 ABC 和 abstractmethod？ | 因为我们要定义一个 `BaseAgent` 抽象基类，所有专业化 Agent 都继承它。这是面向对象设计的标准做法。 |

---

### ➡️ 转场

> 环境就绪。接下来进入核心——先搞定 Agent 之间怎么“说话”。

---

# 第二部分：消息协议与基础架构（00:15–00:30）

---

## 教学段 3：消息协议——Agent 之间的“微信群”

📍 **Cell 范围**：Cell 5（`cell-5`）、Cell 6（`cell-6`）、Cell 7（`cell-7`）

⏱ **时间**：10 分钟

🎯 **目标**：理解多智能体通信协议的设计，掌握 MessageType + Message + BaseAgent 三个核心类

---

### 🗣 话术

> 在 App1 里，Agent 用 Function Calling 调用工具——那是**单向**的，Agent 发命令、工具返回结果。
>
> 但多智能体系统不一样。Planner 要给 Coder 发任务，Coder 做完要给 Reviewer 看，Reviewer 提了意见要反馈给 Coder——这是**双向通信**。
>
> 大家看 Cell 6 的对比表，关键区别：
> - Function Calling：Agent → 工具（单向）
> - 消息传递：Agent ↔ Agent（双向）
>
> 就像公司里的沟通方式：Function Calling 是你对着自动售货机投币取饮料，消息传递是同事之间发微信。
>
> 好，看 Cell 7 的代码。我们定义了三个东西：
>
> **第一，`MessageType` 枚举**——6 种消息类型：
> - `TASK`：分配任务（PM 给工程师发需求）
> - `RESULT`：返回结果（工程师交付代码）
> - `FEEDBACK`：反馈意见（审查员提 Bug）
> - `QUESTION`、`STATUS`、`DELEGATE`：辅助类型
>
> **第二，`Message` 数据类**——每条消息有 sender、receiver、msg_type、content，还有时间戳和唯一 ID。就像微信消息有发送人、接收人、内容、时间。
>
> **第三，`BaseAgent` 抽象基类**——这是所有 Agent 的模板。核心设计：
> - 每个 Agent 有 `inbox`（收件箱）和 `outbox`（发件箱）
> - 有 `llm` 属性，用于调用 LLM 推理
> - 有 `system_prompt`，定义角色身份
> - 有 `process()` 方法要求子类实现——“收到消息后怎么处理”
>
> 运行 Cell 7。没有输出，但这些类已经加载到内存里了。

---

### 👀 输出要点

- Cell 7 运行无输出（纯定义类）
- 学员理解 `MessageType` 有 6 种枚举值
- 学员理解 `Message` 的 `sender/receiver/msg_type/content` 四要素
- 学员理解 `BaseAgent` 是抽象基类，子类必须实现 `process()` 方法

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 为什么不用 JSON 直接通信？ | 结构化的 `Message` 类有类型检查、自动时间戳、唯一 ID，比裸 JSON 更安全、更易调试。生产环境可以序列化成 JSON 传输。 |
| `@dataclass` 和普通 class 的区别？ | `@dataclass` 自动生成 `__init__`、`__repr__` 等方法，减少样板代码。本质还是普通 class。 |
| `BaseAgent` 的 `outbox` 什么时候被清空？ | 等一下看编排器（Cell 15），编排器的 `deliver_messages()` 方法会遍历所有 Agent 的 outbox，投递后清空。 |

---

### ➡️ 转场

> 消息协议有了，接下来看怎么设计每个 Agent 的“人设”。

---

## 教学段 4：System Prompt 设计原则

📍 **Cell 范围**：Cell 8（`cell-8`）

⏱ **时间**：5 分钟

🎯 **目标**：理解多智能体系统中 System Prompt 的四大设计原则

---

### 🗣 话术

> Cell 8 讲了一个非常关键的设计原则：**每个 Agent 都有自己独立的 System Prompt**。
>
> 大家回忆一下 App1——那个 ReAct Agent 只有一个 System Prompt，告诉它“你会用工具解决问题”。
>
> 但多智能体系统里，每个 Agent 必须**只知道自己的职责**。就像公司里，产品经理不需要知道怎么写代码，测试工程师不需要知道怎么做产品规划。
>
> Cell 8 总结了四个原则，大家看表格：
> 1. **角色明确**：告诉 LLM “你是 Reviewer Agent”
> 2. **任务聚焦**：限定范围，“你只负责审查代码，不要自己写代码”
> 3. **输出格式**：规定响应结构，比如 “CORRECTNESS: [PASS/FAIL]”
> 4. **质量标准**：定义评判标准
>
> 这四个原则缺一不可。如果不限定角色，LLM 可能“越权”——Reviewer 自己改代码了；如果不规定输出格式，后续 Agent 解析结果就很困难。
>
> 我们马上就要实现四个 Agent，大家留意每个 Agent 的 System Prompt 是怎么按这四个原则写的。

---

### 👀 输出要点

- 学员记住四个原则：角色明确、任务聚焦、输出格式、质量标准
- 理解为什么 System Prompt 在多智能体系统中比单 Agent 更关键

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 所有 Agent 共享同一个 LLM，不会“串角色”吗？ | 不会。每次调用 LLM 时，System Prompt 不同，LLM 就扮演不同角色。就像同一个演员，给不同剧本就演不同角色。 |
| System Prompt 用中文还是英文？ | 本 notebook 用英文，因为 LLM 对英文指令的遵从性通常更好。生产环境可以根据模型选择。 |

---

### ➡️ 转场

> 原则讲完了，来看真代码。接下来实现四个专业化 Agent。

---

# ☕ 第一次休息（00:30–00:33）

> **三句话回顾前半段：**
> 1. 多智能体系统的核心是“分工协作”——每个 Agent 专注一件事，通过消息传递协作。
> 2. 通信协议由 `MessageType`（6 种消息类型）、`Message`（sender/receiver/content）和 `BaseAgent`（inbox/outbox/process）三个类组成。
> 3. System Prompt 设计四原则：角色明确、任务聚焦、输出格式、质量标准——每个 Agent 的“人设”就靠它。

---

---

# 第三部分：专业化 Agent 实现（00:33–00:50）

---

## 教学段 5：四大 Agent——Planner、Coder、Reviewer、Tester

📍 **Cell 范围**：Cell 9（`cell-9`）、Cell 10（`cell-10`）、Cell 11（`cell-11`）、Cell 12（`cell-12`）、Cell 13（`cell-13`）

⏱ **时间**：17 分钟

🎯 **目标**：理解四个专业化 Agent 的角色设计、System Prompt 和 process() 实现

---

### 🗣 话术

> 好，休息回来。我们要实现四个 Agent，就像组建一个开发团队。
>
> **PlannerAgent（Cell 10）——项目经理**
>
> 大家先看 Cell 10 的 system_prompt：
> ```
> You are the Planner Agent, responsible for breaking down complex software tasks...
> ```
> 它的职责是把复杂任务拆成子任务。比如我让它“写一个 factorial 函数”，它会分析需求、列出步骤：先写核心逻辑、再加错误处理、再写示例。
>
> 关键看 `process()` 方法：收到 TASK 消息后，把任务丢给 LLM，拿到分析结果，然后自动发 TASK 消息给 Coder。**Planner 自己不写代码，只分配任务。**
>
> **CoderAgent（Cell 11）——工程师**
>
> Coder 的 system_prompt 写得很细：“Write clean, readable, well-documented code”、“Include type hints”、“Handle edge cases”。
>
> 它的 `process()` 方法：收到 TASK，调用 LLM 写代码；收到 FEEDBACK（审查意见），再调用 LLM 修改代码。修改后会把新代码发给 Reviewer 继续审查。**这就是反馈回路。**
>
> **ReviewerAgent（Cell 12）——代码审查员**
>
> Reviewer 的 System Prompt 要求它从四个维度审查：正确性、错误处理、代码风格、性能。而且必须输出标准格式：
> ```
> CORRECTNESS: [PASS/FAIL]
> ERROR_HANDLING: [PASS/FAIL]
> ...
> VERDICT: [APPROVE/REVISE]
> ```
> 如果 VERDICT 是 APPROVE，代码传给 Tester；如果是 REVISE，反馈回 Coder 修改。
>
> **TesterAgent（Cell 13）——QA 测试工程师**
>
> Tester 收到代码后，让 LLM 生成测试用例，然后**真正用 `exec()` 执行代码和测试**。这是本 notebook 的亮点——不是假装测试，而是真跑。
>
> 大家注意 Tester 里有一段 `exec(code_to_test, test_namespace)` 和 `exec(test_response, test_namespace)`。先执行被测代码，再执行测试代码。如果 exec 报错，就把错误信息反馈给 Coder。
>
> 好，依次运行 Cell 10、11、12、13。都是纯定义类，没有输出。

---

### 👀 输出要点

- Cell 10–13 运行无输出（纯定义类）
- Planner 的 process()：收到 TASK → LLM 分析 → 发 TASK 给 Coder
- Coder 的 process()：收到 TASK/FEEDBACK → LLM 写/改代码 → 发 RESULT 给 Reviewer
- Reviewer 的 process()：收到 RESULT → LLM 审查 → APPROVE 发给 Tester / REVISE 反馈给 Coder
- Tester 的 process()：收到 RESULT → LLM 生成测试 → exec() 真实执行 → 报告结果

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| `exec()` 执行任意代码不危险吗？ | 在教学环境中可控。生产环境一定要用沙箱（Docker 容器、安全的 subprocess）执行。 |
| 如果 Reviewer 一直 REVISE，会无限循环吗？ | 不会。编排器有 `max_iterations=15` 的限制，超过就自动停止。另外 Planner 也会在多轮后发送 STATUS 消息结束流程。 |
| 四个 Agent 共享一个 LLM 实例？ | 是的。通过不同的 System Prompt，同一个 LLM 扮演不同角色。就像一个会议里同一个人可以先当主持人再当评委。 |
| 为什么 Coder 对 FEEDBACK 和 TASK 的处理不一样？ | TASK 是全新任务，用 Planner 的分析作为上下文；FEEDBACK 是修改意见，用 Reviewer 的反馈作为上下文。输入不同，提示模板也不同。 |

---

### ➡️ 转场

> 四个 Agent 都就位了。但它们还不知道彼此存在——需要一个“调度中心”把它们连起来。

---

## 教学段 6：编排器——多智能体的“调度中心”（00:50–01:00）

📍 **Cell 范围**：Cell 14（`cell-14`）、Cell 15（`cell-15`）

⏱ **时间**：10 分钟

🎯 **目标**：理解 MultiAgentOrchestrator 的消息路由和工作流执行机制

---

### 🗣 话术

> 编排器就像一个快递分拣中心。每个 Agent 把消息放到自己的 outbox，编排器负责取出来、投递到对方的 inbox。
>
> 看 Cell 15 的 `MultiAgentOrchestrator` 类，核心方法有三个：
>
> **1. `register(agent)`**——注册 Agent，把它放进字典里。
>
> **2. `deliver_messages()`**——遍历所有 Agent 的 outbox，把消息投递到接收方的 inbox，然后清空 outbox。就像邮递员挚家挚户取信、送信。
>
> **3. `run(initial_task)`**——这是主循环：
> ```
> 第一步：把初始任务发给 Planner
> 循环（最多 max_iterations 次）：
>     对每个 Agent：如果 inbox 有消息，就 process()
>     deliver_messages() 投递所有新消息
>     如果没有新消息了，说明工作完成，退出
> 最后：从 Planner 的 state 里取最终结果
> ```
>
> 注意两个设计要点：
> - **`max_iterations=15`**：安全阀，防止无限循环。15 轮一般够 2-3 次代码修订了。
> - **消息日志 `message_log`**：所有投递过的消息都存着，方便调试和可视化。
>
> 还有 `get_summary()` 和 `get_transcript()` 两个辅助方法，分别返回统计信息和完整的消息文本记录。
>
> 运行 Cell 15，无输出。

---

### 👀 输出要点

- Cell 15 运行无输出（纯定义类）
- 学员理解 run() 的主循环逻辑：初始化 → 循环（process + deliver）→ 终止条件
- 学员理解 max_iterations 是安全阀
- 学员理解消息路由机制：outbox → deliver → inbox

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 如果消息的 receiver 不存在怎么办？ | `deliver_messages()` 里有判断：`if msg.receiver in self.agents`。不存在就跳过，消息丢弃。 |
| 为什么终止条件是“没有新消息”？ | 如果所有 Agent 的 outbox 都是空的，说明没人需要继续工作了。这是一种优雅的“自然结束”。 |
| 生产环境会用这种同步循环吗？ | 不会。生产环境用消息队列（RabbitMQ、Kafka）做异步投递。但原理一样——路由消息到正确的 Agent。 |

---

### ➡️ 转场

> 基础设施全部就绪——消息协议、四个 Agent、编排器。下面让它们真正跑起来！

---

# 第四部分：运行多智能体系统（01:00–01:15）

---

## 教学段 7：启动流水线——看 AI 团队协作

📍 **Cell 范围**：Cell 16（`cell-16`）、Cell 17（`cell-17`）、Cell 18（`cell-18`）、Cell 19（`cell-19`）

⏱ **时间**：10 分钟

🎯 **目标**：运行完整的多智能体流水线，观察 Planner→Coder→Reviewer→Tester 的协作过程

---

### 🗣 话术

> 激动人心的时刻到了！
>
> **Cell 17**：配置 LLM 后端。默认用 Ollama + qwen3:4b。如果你用 DashScope，取消注释那行。运行后会打印后端信息。
>
> **Cell 18**：创建编排器，注册四个 Agent。运行后你会看到：
> ```
> Registered: PlannerAgent(Planner)
> Registered: CoderAgent(Coder)
> Registered: ReviewerAgent(Reviewer)
> Registered: TesterAgent(Tester)
> ```
> 四个 Agent 都注册好了。
>
> **Cell 19**：核心！我们给系统一个任务：“写一个 factorial 函数，要处理负数异常，要有示例。”
>
> 运行后你会看到一个精彩的协作过程（verbose=True 会打印每一步）：
> 1. **Planner** 收到任务，LLM 分析需求，拆成子任务，发 TASK 给 Coder
> 2. **Coder** 收到任务，LLM 生成代码（一个 factorial 函数），发 RESULT 给 Reviewer
> 3. **Reviewer** 收到代码，LLM 逐项审查——正确性、错误处理、代码风格——输出 APPROVE 或 REVISE
> 4. 如果 APPROVE，代码传给 **Tester**；如果 REVISE，反馈回 Coder 修改
> 5. **Tester** 拿到代码，LLM 生成测试用例，**真正 exec() 执行**，报告结果
>
> 整个过程可能走 5-10 个迭代，每个迭代都有 LLM 调用，大约 30-60 秒跑完。
>
> 大家注意看日志里的 `[Planner -> Coder] (task):`、`[Coder -> Reviewer] (result):` 这些消息流转。
>
> **现在运行 Cell 19！**（等待运行……）
>
> 好，跑完了。大家看日志，是不是完整走了 Planner→Coder→Reviewer→Tester 的流程？有没有出现 Reviewer 打回修改的情况？

---

### 👀 输出要点

- Cell 17 打印 LLM 后端信息（如 `Using Ollama backend with model qwen3:4b`）
- Cell 18 打印 4 条 Registered 信息
- Cell 19 打印完整的协作日志，包含：
  - 各 Agent 的处理过程（`[Agent] processing...`）
  - 消息流转（`[Sender -> Receiver] (type): content`）
  - 最终的迭代次数（`Completed in N iterations`）

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 为什么有时候要跑很多轮？ | Reviewer 可能发现问题要求修改，Coder 修改后再审查，可能来回 2-3 次。这正是多智能体的价值——自动化代码审查。 |
| 输出不确定性怎么办？ | LLM 本质是概率模型，每次输出可能不同。`temperature=0.7` 允许一定创造性。设为 0 则更确定但可能呆板。 |
| 如果 LLM 后端超时？ | 代码里用了 `try/except`，超时会返回错误信息而不是崩溃。但 Ollama 本地一般不会超时。 |

---

### ➡️ 转场

> 流水线跑完了。来看看结果和统计数据。

---

## 教学段 8：结果分析与统计（01:10–01:15）

📍 **Cell 范围**：Cell 20（`cell-20`）、Cell 21（`cell-21`）、Cell 22（`cell-22`）

⏱ **时间**：5 分钟

🎯 **目标**：分析最终结果和工作流统计数据

---

### 🗣 话术

> **Cell 20**：打印最终结果。你会看到 `FINAL RESULT` 下面是 Planner 汇总的最终输出——包含生成的代码和测试结果。
>
> **Cell 21**：看工作流统计。`get_summary()` 返回一个字典，包含：
> - `total_iterations`：总共跑了几轮（通常 5-10 轮）
> - `total_messages`：一共传递了多少条消息
> - `agents`：每个 Agent 分别处理了多少条消息
>
> 大家看看你的数据：
> - Planner 处理了几条？（通常 2-3 条：初始任务 + 子 Agent 的结果汇报）
> - Coder 处理了几条？（如果 Reviewer 打回了，就多几条）
> - 总消息数大概在 8-15 条之间
>
> **Cell 22**：可选，打印完整的消息记录（transcript）。内容很长，可以取消注释看看。
>
> 这就是一个完整的 LLM 驱动的多智能体流水线！

---

### 👀 输出要点

- Cell 20 打印 `FINAL RESULT`，包含最终代码和测试结果
- Cell 21 打印 JSON 格式的统计数据：
  ```json
  {
    "total_iterations": 7,
    "total_messages": 12,
    "agents": {
      "Planner": {"messages_processed": 3},
      "Coder": {"messages_processed": 2},
      "Reviewer": {"messages_processed": 2},
      "Tester": {"messages_processed": 1}
    }
  }
  ```
  （数值会因 LLM 输出而异）

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 为什么 Tester 通常只处理 1 条消息？ | 因为 Tester 是流水线终点，只要代码通过审查就测试一次。除非测试失败反馈回 Coder。 |
| 消息数量和 LLM 调用次数是一样的吗？ | 不完全一样。每个 Agent 的 `process()` 方法可能发多条消息，所以消息数 ≥ LLM 调用数。 |

---

### ➡️ 转场

> 流水线模式搞定了。休息一下，回来学辩论模式。

---

# ☕ 第二次休息（01:15–01:18）

> **三句话回顾：**
> 1. 编排器通过 outbox→deliver→inbox 的消息路由机制连接四个 Agent，主循环最多跑 15 轮。
> 2. 流水线 Planner→Coder→Reviewer→Tester 完整运行，Reviewer 可以打回让 Coder 修改（反馈回路）。
> 3. 统计数据显示了每个 Agent 的消息处理量，总迭代约 5-10 轮、总消息约 8-15 条。

---

---

# 第五部分：辩论模式（01:18–01:30）

---

## 教学段 9：辩论模式——AI 圆桌讨论

📍 **Cell 范围**：Cell 23（`cell-23`）、Cell 24（`cell-24`）、Cell 25（`cell-25`）、Cell 26（`cell-26`）

⏱ **时间**：12 分钟

🎯 **目标**：理解辩论模式的设计，运行一场 AI 辩论，观察多视角分析

---

### 🗣 话术

> 流水线模式适合“顺序处理”的任务。但有些问题没有标准答案——比如“创业公司后端该用 Python 还是 Go？”
>
> 这时候就需要**辩论模式**：让两个 Agent 从对立立场出发，各自用 LLM 推理来辩论，最后由一个**主持人 Agent** 综合评判。
>
> 就像辩论赛：正方、反方、评委。
>
> **Cell 24 定义了两种 Agent：**
>
> **`DebateAgent`**——辩手。System Prompt 里写了它的立场（比如 "arguing in favor of Python"），要求它给出有说服力的论证。
>
> 注意辩手的 `process()` 方法：收到对方观点后，LLM 会**先总结对方论点、再提出反驳**。这就是多轮辩论的核心。
>
> **`ModeratorAgent`**——主持人。它不参与辩论，只负责：
> 1. 开场发出辩题
> 2. 每轮结束后决定“是否继续”
> 3. 最后一轮给出 VERDICT（评判结果）
>
> **Cell 25** 是 `run_debate()` 函数，封装了辩论的编排逻辑。它创建两个辩手 + 一个主持人，注册到编排器，然后 run()。
>
> **Cell 26**：跑一场辩论！题目是“创业公司最佳后端语言”，Python vs Go，3 轮。
>
> 运行后你会看到：
> - Round 1：各自开场陈词
> - Round 2：互相反驳
> - Round 3：最终总结
> - 最后：Moderator 的 VERDICT
>
> **现在运行 Cell 24、25、26！**

---

### 👀 输出要点

- Cell 24、25 运行无输出（纯定义）
- Cell 26 打印辩论过程：
  ```
  ============================================================
  DEBATE: Best backend language for a new startup
  Position A: Python (Django/FastAPI)
  Position B: Go (Gin/Echo)
  Rounds: 3
  ============================================================
  ```
  - 每轮显示 Advocate-A 和 Advocate-B 的发言
  - 最后显示 `VERDICT`：Moderator 的综合评判
- 辩论大约需要 60-90 秒（6 次 LLM 调用 + 评判）

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 辩论结果每次都一样吗？ | 不一样。LLM 每次生成不同论点，结论可能不同。这正好体现了辩论模式的价值——同一个问题可以反复辩论获取多角度分析。 |
| 辩论模式有什么实际用途？ | 技术选型评估、政策分析、产品方案对比。让 AI 扮演不同立场的专家，帮你看到盲区。 |
| 辩手数量可以多于 2 个吗？ | 当然可以。可以加第三个辩手持“中立/折中”立场，或者加“事实核查 Agent”。 |
| 和流水线模式可以结合吗？ | 可以。比如代码审查时用辩论模式：一个 Agent 说“代码没问题”，一个说“这里有隐患”，主持人综合判断。 |

---

### ➡️ 转场

> 两大模式都实现了。最后看看可视化和总结。

---

# 第六部分：可视化与总结（01:30–01:38）

---

## 教学段 10：架构可视化 + 课程总结

📍 **Cell 范围**：Cell 27（`cell-27`）、Cell 28（`cell-28`）、Cell 29（`cell-29`）

⏱ **时间**：8 分钟

🎯 **目标**：通过可视化图表直观理解协作架构，总结关键知识点

---

### 🗣 话术

> **Cell 28** 用 matplotlib 画了多智能体协作的架构图。运行后你会看到两幅图：
> - **左图**：流水线模式——Planner→Coder→Reviewer→Tester，带一条 Reviewer→Coder 的反馈回路箭头
> - **右图**：辩论模式——Moderator 在中间，两个 Advocate 在两侧
>
> 这两幅图非常适合放在 PPT 或文档里解释多智能体架构。
>
> **Cell 29 是总结**。大家看关键模式的对比表：
>
> | 模式 | 适用场景 |
> |:---|:---|
> | 流水线 | 顺序处理（软件开发、内容生产） |
> | 监督者 | 协同调度（Planner 协调） |
> | 辩论 | 多视角分析（技术选型、方案评估） |
> | 反馈回路 | 质量提升（Reviewer 打回修改） |
>
> 生产环境注意事项五条，大家看 Cell 29：
> 1. **持久化**：消息和状态存数据库
> 2. **异步处理**：用消息队列
> 3. **错误处理**：Agent 失败时优雅降级
> 4. **监控**：记录所有交互用于调试
> 5. **成本控制**：每个 Agent 设 token 预算
>
> 最后讲适用场景：软件开发、内容生产、决策支持、客户服务。

---

### 👀 输出要点

- Cell 28 输出两幅架构图（matplotlib 渲染）
- Cell 29 无输出（纯 Markdown 总结）
- 学员理解四种模式的适用场景
- 学员知道生产环境五大注意事项

---

### ❓ 预判 Q&A

| 问题 | 回答 |
|:---|:---|
| 这种架构和微服务有什么关系？ | 非常像！每个 Agent 就是一个微服务，编排器就是 API Gateway / Service Mesh，消息传递就是事件驱动。 |
| Token 成本怎么控制？ | 每次 LLM 调用都消耗 token。可以给每个 Agent 设上限，超过就停止或切换到更便宜的模型。 |
| 可以用不同的 LLM 给不同 Agent 吗？ | 当然可以。比如 Planner 用大模型（GPT-4 级别）负责规划，Coder 用 Code 专用模型，Reviewer 用小模型。 |

---

### ➡️ 转场

> 好，今天的内容就到这里。我们来做一个小练习巩固一下。

---

# 练习与扩展

---

## 练习 1：修改辩论主题

**任务**：把 Cell 26 的辩论题目换成你感兴趣的技术选型，比如 "React vs Vue"、"SQL vs NoSQL" 或 "Monolith vs Microservices"。

**提示**：
1. 只需修改 `topic`、`position_a`、`position_b` 三个参数
2. 试试调整 `max_rounds`（2 轮 vs 5 轮），观察辩论深度变化
3. 注意每多一轮，LLM 调用次数增加 2 次（两个辩手各一次）

**验证**：运行后能看到完整的辩论日志和 VERDICT。

---

## 练习 2：添加第五个 Agent——DocumenterAgent

**任务**：创建一个 DocumenterAgent，在 Tester 之后自动生成 docstring 文档。

**提示**（逐步揭示）：
1. 继承 `BaseAgent`，给它一个 System Prompt："You are the Documenter Agent, responsible for writing comprehensive docstrings..."
2. `process()` 方法：收到 RESULT → 让 LLM 为代码添加 docstring → 发 RESULT 给 Planner
3. 在编排器中用 `orchestrator.register(DocumenterAgent(llm))` 注册
4. 需要修改 TesterAgent 的 process()，让它测试通过后发消息给 Documenter 而不是 Planner

**常见错误**：
- 忘记继承 `BaseAgent`
- `process()` 方法签名写错（参数是 `message: Message`）
- 新 Agent 的名字和已注册的重复
- 忘记在 Tester 里修改接收方

**验证**：运行后在 transcript 中能看到 `[Tester -> Documenter]` 和 `[Documenter -> Planner]` 的消息。

---

## 练习 3（进阶）：实现投票机制

**任务**：创建 3 个 ReviewerAgent（不同审查重点），用投票决定是否通过。

**提示**：
1. 创建 SecurityReviewer、PerformanceReviewer、StyleReviewer
2. 让 Coder 同时发消息给 3 个 Reviewer
3. 在 Planner 中收集 3 个审查结果，多数决定通过
4. 这是“群体模式”的雏形

---

---

# 附录

---

## 时间表速查

| 时间段 | 内容 | Cell |
|:---|:---|:---|
| 00:00–00:08 | 开场 + 四大模式概览 | 0–1 |
| 00:08–00:15 | 环境配置 | 2–4 |
| 00:15–00:30 | 消息协议 + BaseAgent + Prompt 设计 | 5–8 |
| 00:30–00:33 | ☕ 休息 1 | — |
| 00:33–00:50 | 四大 Agent 实现 | 9–13 |
| 00:50–01:00 | 编排器 | 14–15 |
| 01:00–01:15 | 运行流水线 + 结果分析 | 16–22 |
| 01:15–01:18 | ☕ 休息 2 | — |
| 01:18–01:30 | 辩论模式 | 23–26 |
| 01:30–01:38 | 可视化 + 总结 | 27–29 |

---

## 关键数据速查

| 数据项 | 典型值 |
|:---|:---|
| 源码 Cell 总数 | 30（14 Markdown + 16 Code） |
| 专业化 Agent 数量 | 4（Planner、Coder、Reviewer、Tester） |
| MessageType 枚举值 | 6（TASK、RESULT、QUESTION、FEEDBACK、STATUS、DELEGATE） |
| 编排器 max_iterations 默认值 | 15 |
| 流水线典型迭代数 | 5–10 轮 |
| 流水线典型消息数 | 8–15 条 |
| 辩论默认轮数 | 3 轮 |
| 辩论 LLM 调用次数 | 约 6–8 次（每轮 2 辩手 + 评判） |
| 单次 LLM 调用耗时（Ollama 本地） | 约 3–8 秒 |
| 流水线总运行时间 | 约 30–90 秒 |
| 辩论总运行时间 | 约 30–60 秒 |

---

## 应急预案

| 问题 | 解决方案 |
|:---|:---|
| Ollama 连接失败 | 检查 `ollama serve` 是否运行；重启 Ollama 服务 |
| LLM 响应超时 | Ollama 首次加载模型较慢，等待 30 秒；或换用 DashScope 云端 |
| 代码运行中 exec() 报错 | 正常现象——这是 Tester 在执行测试，错误信息会反馈给 Coder 修改 |
| 迭代次数达到 15 上限 | 编排器会自动停止；可以 `max_iterations=20` 放宽限制 |
| Reviewer 一直打回 REVISE | LLM 对格式输出不稳定导致；重跑一次通常解决；或降低 temperature |
| 辩论结果偏向一方 | 正常——LLM 对某些话题有倾向；可以交换 position_a 和 position_b 再跑一次对比 |
| matplotlib 中文显示方块 | 检查系统是否有 Microsoft YaHei 或 SimHei 字体；或 `pip install matplotlib` 更新 |
| ModuleNotFoundError | 检查 Python 环境和 `sys.path`；运行 `pip install -r requirements.txt` |

---

## 核心架构对比

| 维度 | App1 (ReAct) | App4 (Multi-Agent) |
|:---|:---|:---|
| Agent 数量 | 1 | 4+（可扩展） |
| 通信方式 | Function Calling（Agent→工具） | 消息传递（Agent↔Agent） |
| 协调者 | Agent 自己决定 | Orchestrator 统一调度 |
| 工作流 | 单 Agent 循环（Think→Act→Observe） | 多 Agent 流水线 + 反馈回路 |
| 适用复杂度 | 简单任务 | 复杂协作任务 |